# Tutorial 0 — A Guided Tour of the Hugging Face Ecosystem

## 0. Mental model: five libraries, one workflow

```text
                    huggingface_hub
      search / download / upload models, datasets, Spaces
                           |
        -------------------------------------
        |              |                    |
   transformers     datasets            evaluate
  tokenizer/model   load & slice   score predictions
  pipeline/generate  real data     against ground truth   
        |
    accelerate
  runs it on CPU/GPU/multi-GPU without code changes
```

Rule of thumb for *which library owns which job*:

| Question | Library |
|---|---|
| "Which models/datasets exist, and what are their licenses?" | `huggingface_hub` |
| "How do I turn text into a model and back?" | `transformers` |
| "Where do I get labeled data to test or train on?" | `datasets` |
| "Is my model actually any good?" | `evaluate` |
| "How do I run this faster / on more hardware?" | `accelerate` |
| "How do I let other people try my model in a browser?" | Spaces |

## 1. Install the libraries

In [1]:
!pip -q install -U transformers accelerate huggingface_hub datasets evaluate sentencepiece ipywidgets


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 3.3.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.17.0 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Code walkthrough — installation command

```bash
!pip -q install -U transformers accelerate huggingface_hub datasets evaluate sentencepiece ipywidgets
```

This line runs a shell command from inside the notebook.

- **`!`**: Jupyter/Colab syntax for executing a system shell command instead of Python code.
- **`pip`**: Python's package installer.
- **`-q` / `--quiet`**: reduces installation output. This keeps the notebook readable; remove it when debugging installation problems.
- **`install`**: tells `pip` to install packages.
- **`-U` / `--upgrade`**: upgrades each package if an older version is already installed. 
- **`transformers`**: model architectures, tokenizers, generation utilities, `Trainer`, and the high-level `pipeline()` API.
- **`accelerate`**: hardware/device orchestration for CPU, GPU, multi-GPU, mixed precision, and distributed execution.
- **`huggingface_hub`**: programmatic access to repositories and metadata on the Hugging Face Hub.
- **`datasets`**: loading, transforming, slicing, streaming, and caching datasets.
- **`evaluate`**: standardized evaluation metrics such as accuracy, F1, BLEU, and ROUGE.
- **`sentencepiece`**: tokenizer backend required by many models, especially multilingual and encoder-decoder models.
- **`ipywidgets`**: interactive notebook controls used later for dropdowns and text boxes.


## 2. Search the Hub before you download anything

The Hub hosts models, datasets, and Spaces. You can query metadata (downloads, license, task, size)
without pulling any weights — useful for choosing between candidates quickly.

![The Hugging Face model browser at huggingface.co/models, with the Tasks filter, name filter, and sort control highlighted](images/huggingface_overview.png)

The web UI at [huggingface.co/models](https://huggingface.co/models) exposes exactly the same filters the
API call below uses: the **Tasks** sidebar is `pipeline_tag=`, **Filter by name** is `search=`, and the
**Sort** control is `sort=`. Browsing is good for getting a feel for what exists; the API call is what you
reach for when you want a reproducible shortlist inside a script.


In [2]:
from huggingface_hub import list_models

candidates = list_models(
    pipeline_tag="text-classification",
    search="sentiment",
    sort="downloads",
    limit=5,
)

for m in candidates:
    print(f"{m.id:55s} downloads={m.downloads:>10,}  likes={m.likes}")


cardiffnlp/twitter-roberta-base-sentiment-latest        downloads= 3,147,116  likes=831
pysentimiento/robertuito-sentiment-analysis             downloads= 1,111,400  likes=104
lxyuan/distilbert-base-multilingual-cased-sentiments-student downloads= 1,077,704  likes=316
cardiffnlp/twitter-xlm-roberta-base-sentiment           downloads=   986,374  likes=276
nlptown/bert-base-multilingual-uncased-sentiment        downloads=   733,647  likes=485


### Code walkthrough — `list_models(...)` argument by argument

`list_models()` queries **metadata on the Hub**. It does not download model weights.

```python
candidates = list_models(
    pipeline_tag="text-classification",
    search="sentiment",
    sort="downloads",
    limit=5,
)
```

- **`pipeline_tag="text-classification"`**: restricts results to repositories tagged for text classification. This prevents unrelated repositories from dominating the search.
- **`search="sentiment"`**: performs a text search over model metadata/name information. Here it narrows classification models toward sentiment models.
- **`sort="downloads"`**: asks the Hub to rank matching repositories by download count.
- **`limit=5`**: return at most five matches. Keeping this small is useful for an initial shortlist.

`list_models()` returns an **iterable of model metadata objects**, not instantiated neural networks. For each result `m`:

- **`m.id`** is the repository identifier, e.g. `organization/model-name`.
- **`m.downloads`** is Hub download metadata.
- **`m.likes`** is the number of Hub likes.

The formatted print statement also contains useful Python formatting:

```python
f"{m.id:55s} downloads={m.downloads:>10,}  likes={m.likes}"
```

- **`:55s`** reserves a 55-character field for the string so rows line up.
- **`:>10,`** right-aligns the number in a 10-character field and inserts thousands separators.

**Important distinction:** popularity is a discovery signal, not evidence that a model is best for your data. Model card, license, language, architecture, size, and evaluation setup still matter.


### Predict before you run

Before running the next cell: do you expect `Private` and `Gated` to be booleans, or something else?
And will `siblings` (the file list) include the model card (`README.md`)?

In [ ]:
from huggingface_hub import model_info

MODEL_ID = "distilbert-base-uncased-finetuned-sst-2-english"
info = model_info(MODEL_ID)

print("Model ID:       ", info.id)
print("Private:        ", info.private)
print("Gated:          ", info.gated)
print("Pipeline tag:   ", info.pipeline_tag)
print("Library:        ", info.library_name)
print("License tag:    ", [t for t in (info.tags or []) if t.startswith("license:")])
print("Downloads (30d):", f"{info.downloads:,}")
print("Likes:          ", info.likes)
print("Last modified:  ", info.last_modified)

# Parameter count is only reported for repos that ship .safetensors weights.
safetensors = getattr(info, "safetensors", None)
if safetensors is not None:
    print("Parameters:     ", f"{safetensors.total:,}")

print("Files:")
for s in info.siblings:
    print(" -", s.rfilename)


### Code walkthrough — `model_info(...)` and repository metadata

```python
MODEL_ID = "distilbert-base-uncased-finetuned-sst-2-english"
info = model_info(MODEL_ID)
```

- **`MODEL_ID`**: the Hub repository identifier. It may be a simple name for an official/global repository or an `owner/repository` name.
- **`model_info(MODEL_ID)`**: requests repository metadata for that model. No weights are downloaded and no model is instantiated here.

### The same repository, two views

Everything the cell printed is also visible by eye on the model's page on the Hub:

![The model card page for distilbert-base-uncased-finetuned-sst-2-english on the Hugging Face Hub](images/distilbert_base.png)

| On the model page | In the `info` object |
|---|---|
| the repository title, `distilbert/distilbert-base-uncased-finetuned-sst-2-english` | `info.id` |
| the `Text Classification` tag | `info.pipeline_tag` |
| the `Transformers` tag | `info.library_name` |
| the `License: apache-2.0` tag | the `license:apache-2.0` entry in `info.tags` |
| **Downloads last month** | `info.downloads` |
| the **Like** counter | `info.likes` |
| **Model size — 67M params** | `info.safetensors.total` |
| the **Files and versions** tab | `info.siblings` |
| no padlock and no access form on the page | `info.private`, `info.gated` |

The page is for browsing; `model_info()` is the same facts in a form a script can branch on — which is what
you want when you are checking twenty candidates, or asserting a license before a training run.

### Field by field

- **`info.id`**: canonical repository ID.
- **`info.private`**: a genuine boolean — whether the repository is visible only to you or your organization.
- **`info.gated`**: **not** a boolean. It is `False` for an ungated repo, or the string `"auto"` / `"manual"` when users must accept terms before downloading files. Test it with `if info.gated:` rather than comparing against `True`.
- **`info.pipeline_tag`**: the task associated with the repository, such as `text-classification`. This is what `pipeline()` consults when you do not name a task explicitly.
- **`info.library_name`**: the framework the repo declares, e.g. `transformers`.
- **`info.tags`**: metadata tags. The list comprehension
  ```python
  [t for t in (info.tags or []) if t.startswith("license:")]
  ```
  uses **`info.tags or []`** so the code still works if `tags` is missing/`None`, and **`startswith("license:")`** keeps only license-related tags. Licenses live in this same flat tag list alongside task, language, and dataset tags.
- **`info.downloads`**: downloads over the last 30 days — the same number the page shows, so it is recent activity, not a lifetime total.
- **`info.likes`**: number of Hub likes.
- **`info.last_modified`**: a `datetime` for the most recent commit. A checkpoint untouched for years is not automatically bad, but it is worth noticing.
- **`info.safetensors`**: present only when the repo ships `.safetensors` weights; **`.total`** is the parameter count. The screenshot's `67M params` comes from exactly this field, which is why the code guards it with `getattr(...)` — ask for it on a repo that has only `.bin` weights and you would get `None`.
- **`info.siblings`**: repository files as metadata objects, with **`s.rfilename`** giving each file's path inside the repo.

### What metadata cannot tell you

The left-hand side of the page — **Model Details**, **Uses**, **Risks, Limitations and Biases**, **Training** —
is the `README.md` you will find in `info.siblings`, and it is prose written by humans. The API can tell you
that this checkpoint is Apache-2.0, English, 67M parameters, and fine-tuned for text classification. Only the
card text tells you that it was fine-tuned on SST-2, reaches 91.3 accuracy on that dev set, and carries
documented bias caveats.

Metadata narrows the field; the card decides. That is the habit worth forming: filter by API, then read the
card of the two or three that survive.


### Exercise 2.1

Use `list_models(task=..., search=..., sort="downloads", direction=-1, limit=5)` to find the top-5 most
downloaded models for `"summarization"` and for `"translation"`. For each, note the model ID and whether
it looks like a fine-tuned checkpoint or a general-purpose base model (the name is usually a strong hint).

## 3. `transformers`: the same four-step recipe, every time

Every `transformers` task follows the same shape, no matter which model or task:

```text
text  →  tokenizer  →  model  →  post-processing  →  usable output
```

`pipeline()` bundles all four steps for you. Below we run **three different tasks** with three different
model architectures, using the exact same `pipeline()` call shape, to show how much the library
standardizes for you.

![How pipeline() works: model and tokenizer files download once from the Hub into a local cache, then raw text flows through tokenizer, model forward pass, logits, softmax and label mapping to a prediction](images/pipeline2.png)

Two things in that diagram are worth pausing on, because they routinely surprise people.

**The download happens once.** The first `pipeline(...)` call pulls the model and tokenizer files from the Hub
into a local cache (`~/.cache/huggingface/hub/`, or wherever `HF_HOME` points). Every later call reads from
that cache, which is why the first cell below is slow and the rest are quick. On Colab that cache sits on an
ephemeral VM, so a fresh runtime downloads everything again.

**Inference runs on your machine.** `pipeline()` is not a call to a web API. Once the files are cached, the
forward pass happens on your own CPU or GPU and your input text never leaves the machine — which matters as
soon as the text you want to classify is not yours to hand to a third party.

The bottom row also names the step the sketch above glosses as "post-processing": the model emits raw
**logits**, softmax turns them into probabilities, and the label mapping turns the winning index into a name
like `POSITIVE`. That is where the `score` in the output below comes from.


### 3.1 Sentiment analysis

In [4]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

for text in [
    "This course finally made transformers click for me.",
    "I've spent three hours debugging a shape mismatch and I regret everything.",
]:
    print(text, "->", sentiment(text))


OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\Asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

### Code walkthrough — creating and calling a sentiment pipeline

```python
sentiment = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)
```

`pipeline()` builds a high-level inference object that usually handles tokenization, batching, model execution, and task-specific post-processing.

- **first positional argument: `"sentiment-analysis"`**: the task. It tells Transformers which pipeline class and output interpretation to use. For this task, output is normally a label plus a confidence score.
- **`model=...`**: exact model repository/checkpoint to load. Providing it makes the example reproducible and avoids relying on whichever default model a library version happens to choose.

No tokenizer is supplied explicitly. `pipeline()` therefore loads the tokenizer associated with the selected checkpoint when possible.

Later:

```python
sentiment(text)
```

- **`text`** is the input string passed to the pipeline's `__call__` method.
- The pipeline tokenizes the string, runs the model, converts logits to task-level scores, and maps the winning class ID to a human-readable label.

The loop simply calls the same already-loaded pipeline on two strings. The model is *not* reloaded for every iteration.

**Useful optional arguments you may see later:** `device`, `batch_size`, `tokenizer`, and model-specific loading arguments. They are intentionally omitted here to keep the first example simple.


### 3.2 Translation

### Predict before you run

**Question:** the model below is `Helsinki-NLP/opus-mt-en-fr` — a model *specific to one language pair*.
Do you expect it to also handle English→German if you just change the input text? Why or why not?

In [ ]:
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-en-fr")

result = translator("Hugging Face makes it easy to share and reuse machine learning models.")
print(result[0]["translation_text"])


### Code walkthrough — translation pipeline

```python
translator = pipeline(
    "translation",
    model="Helsinki-NLP/opus-mt-en-fr"
)
```

- **`"translation"`**: selects the sequence-to-sequence translation pipeline.
- **`model="Helsinki-NLP/opus-mt-en-fr"`**: loads a checkpoint trained for **English → French**. The language pair is part of what the checkpoint has learned; changing the input sentence does not magically change the target language.

Then:

```python
result = translator("Hugging Face makes it easy ...")
```

The single positional argument is the source text. The returned value is a **list of result dictionaries**, even for one input. Therefore:

- **`result[0]`**: first returned translation item.
- **`["translation_text"]`**: extracts only the translated string from that result dictionary.

Under the hood, this differs from classification: an encoder-decoder model generates a sequence token by token rather than choosing among fixed labels.


### 3.3 Text generation

In [ ]:
generator = pipeline("text-generation", model="Qwen/Qwen3-0.6B")

out = generator(
    "The three main things I want to remember about the Hugging Face ecosystem are:",
    max_new_tokens=60,
    do_sample=False,
)
print(out[0]["generated_text"])


### Code walkthrough — text generation arguments

```python
generator = pipeline("text-generation", model="Qwen/Qwen3-0.6B")
```

- **`"text-generation"`**: selects causal/autoregressive language generation.
- **`model="Qwen/Qwen3-0.6B"`**: chooses the exact checkpoint. `0.6B` indicates a relatively small model size, which makes it more practical for a tutorial.

Generation is then invoked as:

```python
out = generator(
    PROMPT,
    max_new_tokens=60,
    do_sample=False,
)
```

- **first positional argument (`PROMPT`)**: the text context given to the model. The model predicts continuations conditioned on this prefix.
- **`max_new_tokens=60`**: upper bound on the number of *new* tokens generated after the prompt. This is usually easier to reason about than `max_length`, which counts prompt + continuation together.
- **`do_sample=False`**: disables stochastic token sampling. With the usual single-beam defaults this makes decoding greedy: at each step the highest-scoring next token is selected. This is useful for reproducible demonstrations.

The output is again a list of dictionaries:

- **`out[0]`**: first generated sequence.
- **`["generated_text"]`**: text returned by the pipeline. For a standard text-generation pipeline, this commonly includes the original prompt followed by the generated continuation.

If **`do_sample=True`**, parameters such as `temperature`, `top_p`, and `top_k` become especially important because they control randomness and diversity.


### Exercise 3.1

Pick one more task from the [pipeline task list](https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.pipeline.task)
— e.g. `"zero-shot-classification"`, `"summarization"`, or `"question-answering"` — and run it on your own
input. Note the model it downloaded by default (check `pipeline(task).model.name_or_path` or the download
log).

## 4. `datasets`: where the data comes from

Models are only half the story — you'll constantly need labeled data to evaluate or fine-tune on.
The `datasets` library gives you a uniform interface to thousands of datasets on the Hub, with
memory-mapped, streaming-friendly loading so you don't need to fit everything in RAM.

In [ ]:
from datasets import load_dataset

reviews = load_dataset("rotten_tomatoes", split="test[:50]")

print(reviews)
print()
print("Features:", reviews.features)
print()
print("First example:", reviews[0])


### Code walkthrough — `load_dataset(...)` and dataset slicing

```python
reviews = load_dataset("rotten_tomatoes", split="test[:50]")
```

- **first positional argument: `"rotten_tomatoes"`**: dataset repository/builder name.
- **`split="test[:50]"`**: asks for only the first 50 examples of the test split. Hugging Face Datasets supports split expressions such as:
  - `"train"` → entire training split;
  - `"train[:100]"` → first 100 examples;
  - `"train[:10%]"` → first 10%;
  - `"train[10%:20%]"` → a percentage slice.

Using a 50-row slice is intentional: it keeps the tutorial fast while preserving the same API you would use for a larger evaluation.

The next expressions demonstrate the `Dataset` interface:

- **`reviews.features`**: schema/feature types for all columns.
- **`reviews[0]`**: one row as a Python dictionary.
- `print(reviews)` shows metadata such as column names and number of rows.

A key benefit of `datasets` is that you work with a typed dataset object rather than manually parsing raw files.


### Predict before you run

`reviews.features["label"]` describes the label column. **Question:** given that this is a movie-review
sentiment dataset, do you expect the label to be stored as a raw integer, a string, or a `ClassLabel`
(an integer with named classes attached)? Run the next cell to check.

In [ ]:
print(reviews.features["label"])
print("Class names:", reviews.features["label"].names)


### Code walkthrough — `ClassLabel` metadata

```python
reviews.features["label"]
```

- **`reviews.features`** is a mapping from column name to feature specification.
- **`["label"]`** selects the schema for the label column.

For a classification dataset, this is often a `ClassLabel` object rather than a bare integer type. The stored examples can therefore use compact integer IDs while the schema preserves their semantic names.

```python
reviews.features["label"].names
```

- **`.names`** returns the ordered class-name list.
- The list position is important: index `0` corresponds to class ID `0`, index `1` to class ID `1`, etc.

This is safer than assuming that `0` and `1` always mean the same thing across datasets.


In [ ]:
import pandas as pd

pd.DataFrame(reviews[:5])


### Code walkthrough — converting a small slice to pandas

```python
pd.DataFrame(reviews[:5])
```

- **`reviews[:5]`** uses dataset slicing to retrieve the first five rows. For a Hugging Face `Dataset`, a slice returns a dictionary of columns containing lists of values.
- **`pd.DataFrame(...)`** converts that column-oriented dictionary into a familiar tabular pandas view.

Why only five rows? Converting a tiny slice is convenient for inspection. Converting a very large dataset to pandas can consume substantial RAM and removes some advantages of the `datasets` format.


## 5. Putting it together: score a real model on real data with `evaluate`

This is the workflow you'll use constantly: run a model over a dataset, then use `evaluate` to turn raw
predictions into a trustworthy number instead of eyeballing a handful of examples.

In [ ]:
predictions = sentiment([ex["text"] for ex in reviews])
pred_labels = [0 if p["label"] == "NEGATIVE" else 1 for p in predictions]
true_labels = reviews["label"]

print("First 5 predictions:", pred_labels[:5])
print("First 5 true labels:", true_labels[:5])


### Code walkthrough — batch inference and label alignment

```python
predictions = sentiment([ex["text"] for ex in reviews])
```

The list comprehension extracts only the text field from every dataset example. Passing a **list of strings** to the pipeline requests inference for multiple examples rather than calling it manually one row at a time.

```python
pred_labels = [0 if p["label"] == "NEGATIVE" else 1 for p in predictions]
```

This converts the model's string labels into the integer encoding expected by the dataset:

- **`p["label"]`**: model's predicted class name.
- **`0 if ... else 1`**: explicit mapping from `NEGATIVE` to `0` and everything else here to `1`.

This mapping is crucial: metrics compare IDs numerically, so model-output labels and dataset labels must use the same convention.

```python
true_labels = reviews["label"]
```

- Passing a **column name** instead of an integer row index returns the complete label column.

Finally, `[:5]` is only for displaying a small sanity-check sample. It does not alter the full lists used for evaluation.


### Predict before you run

**Question:** `distilbert-base-uncased-finetuned-sst-2-english` was fine-tuned specifically for sentiment
classification. Do you expect accuracy on this *different* movie-review dataset to be close to what you'd
see on its original training data, clearly lower, or about the same? This is the difference between
in-distribution and out-of-distribution evaluation.

In [ ]:
import evaluate

accuracy_metric = evaluate.load("accuracy")
result = accuracy_metric.compute(predictions=pred_labels, references=true_labels)

print("Accuracy on 50 rotten_tomatoes test examples:", result["accuracy"])


### Code walkthrough — loading and computing a metric

```python
accuracy_metric = evaluate.load("accuracy")
```

- **`"accuracy"`**: metric identifier. `evaluate.load()` returns the metric implementation together with its expected input format.

```python
result = accuracy_metric.compute(
    predictions=pred_labels,
    references=true_labels,
)
```

- **`predictions=pred_labels`**: model outputs after converting them to the dataset's label IDs.
- **`references=true_labels`**: ground-truth labels. `references` is the standard Evaluate term for gold/target answers.

For ordinary classification accuracy, the metric is conceptually

\[
\mathrm{Accuracy}=\frac{\#\{i:\hat y_i=y_i\}}{N}.
\]

The result is a dictionary, so **`result["accuracy"]`** extracts the numeric score.

Using named arguments (`predictions=...`, `references=...`) makes the role of each array explicit and reduces the chance of accidentally swapping them.


In [ ]:
# Self-check: accuracy must be a valid probability.
assert 0.0 <= result["accuracy"] <= 1.0, "Accuracy out of range — something is wrong with the label mapping."
print("OK: accuracy is a valid fraction.")


### Code walkthrough — defensive validation with `assert`

```python
assert 0.0 <= result["accuracy"] <= 1.0, \
    "Accuracy out of range — something is wrong with the label mapping."
```

Python's `assert` has two parts:

1. **condition:** `0.0 <= result["accuracy"] <= 1.0`
2. **message:** displayed if the condition is false.

The chained comparison is equivalent to checking both `accuracy >= 0.0` and `accuracy <= 1.0`.

This is a **sanity check**, not a complete correctness proof. A wrong label mapping could still produce a value between 0 and 1. Its purpose is to catch impossible outputs quickly during a tutorial.


### Exercise 5.1

`evaluate` also has `precision`, `recall`, and `f1`. Load one of them and compute it on the same
predictions. Does it tell a different story than raw accuracy — especially if the two classes aren't
perfectly balanced in this 50-row slice? Check the class balance with
`pd.Series(true_labels).value_counts()` first.

## 6. Interactive task explorer

Pick a task and type your own input — see the ecosystem's breadth without writing new code for each
task.

In [ ]:
import ipywidgets as widgets

_pipelines_cache = {}

def get_pipeline(task):
    if task not in _pipelines_cache:
        model_for_task = {
            "sentiment-analysis": "distilbert-base-uncased-finetuned-sst-2-english",
            "translation_en_to_fr": "Helsinki-NLP/opus-mt-en-fr",
            "text-generation": "Qwen/Qwen3-0.6B",
        }[task]
        real_task = "translation" if task.startswith("translation") else task
        _pipelines_cache[task] = pipeline(real_task, model=model_for_task)
    return _pipelines_cache[task]

def run_task(task, text):
    pipe = get_pipeline(task)
    if task == "text-generation":
        out = pipe(text, max_new_tokens=40, do_sample=False)
        print(out[0]["generated_text"])
    elif task.startswith("translation"):
        print(pipe(text)[0]["translation_text"])
    else:
        print(pipe(text))

widgets.interact(
    run_task,
    task=widgets.Dropdown(
        options=["sentiment-analysis", "translation_en_to_fr", "text-generation"],
        value="sentiment-analysis",
    ),
    text=widgets.Text(value="Hugging Face is a great place to start with LLMs.", layout=widgets.Layout(width="500px")),
)


### Code walkthrough — interactive task explorer

This cell combines **caching**, **task routing**, and **Jupyter widgets**.

#### 1. Pipeline cache

```python
_pipelines_cache = {}
```

A dictionary stores already-loaded pipelines. Model loading is expensive, so the notebook should not reload a checkpoint every time the user changes text.

```python
def get_pipeline(task):
```

- **`task`**: string selected by the widget, e.g. `"sentiment-analysis"`.

```python
if task not in _pipelines_cache:
```

Only create the pipeline the first time that task is requested.

```python
model_for_task = {...}[task]
```

The dictionary maps each UI task choice to a specific model ID. Indexing with `[task]` selects the matching checkpoint.

```python
real_task = "translation" if task.startswith("translation") else task
```

The UI uses `translation_en_to_fr` as a descriptive name, but Transformers expects the pipeline task name `translation`; this line converts the UI label into the API task.

```python
pipeline(real_task, model=model_for_task)
```

- **`real_task`**: actual Transformers pipeline task.
- **`model=model_for_task`**: checkpoint selected from the mapping.

#### 2. Running the selected task

```python
def run_task(task, text):
```

- **`task`**: current dropdown value.
- **`text`**: current text-box content.

For generation:

- **`max_new_tokens=40`** limits continuation length.
- **`do_sample=False`** keeps generation deterministic/greedy under the usual default decoding setup.

For translation, `[0]["translation_text"]` extracts the first translated string. Other tasks print the pipeline result directly.

#### 3. `widgets.interact(...)`

```python
widgets.interact(run_task, task=..., text=...)
```

- **first argument `run_task`**: callback function to execute whenever an input widget changes.
- **`task=widgets.Dropdown(...)`**: binds the function's `task` parameter to a dropdown.
- **`text=widgets.Text(...)`**: binds the function's `text` parameter to a text box.

`Dropdown` arguments:

- **`options=[...]`**: allowed task values shown to the user.
- **`value="sentiment-analysis"`**: initial selection; it must be one of `options`.

`Text` arguments:

- **`value=...`**: initial text shown in the input field.
- **`layout=widgets.Layout(width="500px")`**: presentation-only setting controlling widget width.

The important design pattern is that the UI is thin: the actual model logic remains in ordinary Python functions.


## 7. Spaces: sharing a running demo, not just weights

A **Space** is a small hosted app (Gradio, Streamlit, or a static site) that wraps a model so anyone can
try it in a browser — no local setup. You won't build one in this notebook, but it's worth knowing when
you'd reach for one:

- a model checkpoint alone requires the visitor to have Python + `transformers` installed;
- a Space wraps that same model in a small UI and runs it on Hugging Face's infrastructure (or your own).

The minimal shape of a Gradio-based Space is:

```python
import gradio as gr
from transformers import pipeline

pipe = pipeline("sentiment-analysis")

def classify(text):
    return pipe(text)[0]

gr.Interface(fn=classify, inputs="text", outputs="json").launch()
```

You push that `app.py` (plus a `requirements.txt`) to a Space repo the same way you'd push a model with
`huggingface_hub`, and the Hub builds and hosts it for you.


### Code walkthrough — the Gradio Space example

The embedded example has four important API calls:

```python
pipe = pipeline("sentiment-analysis")
```

- **`"sentiment-analysis"`** selects the Transformers task.
- No **`model=`** argument is supplied, so the library chooses its configured/default model for that task. For reproducible applications, explicitly specifying a model ID is usually better.

```python
def classify(text):
    return pipe(text)[0]
```

- **`text`** is the value supplied by the web UI.
- **`pipe(text)`** runs model inference.
- **`[0]`** extracts the first result dictionary because pipelines commonly return a list, even for one input.

```python
gr.Interface(fn=classify, inputs="text", outputs="json")
```

- **`fn=classify`**: Python function Gradio should call.
- **`inputs="text"`**: create a text input component and pass its value to `classify`.
- **`outputs="json"`**: render the returned Python dictionary/list as JSON-like structured output.

```python
.launch()
```

Starts the Gradio app. In a Hugging Face Space, the hosting environment runs the app process and exposes the interface in the browser.

A production Space would usually also pin dependencies and model versions, handle errors, and possibly configure authentication or GPU hardware.


## 8. Where each later tutorial fits

| Topic | Library | Notebook |
|---|---|---|
| Tokenization deep dive | `transformers` (tokenizers) | Tutorial 2 |
| Transformer internals: hidden states, attention, KV cache | `transformers` | Tutorial 3 |
| Fine-tuning a model on your own data | `transformers` + `datasets` + `Trainer` | later tutorial |
| Running on multiple GPUs / mixed precision | `accelerate` | later tutorial |
| Efficient inference: quantization, batching, FlashAttention | `transformers` + `accelerate` | later tutorial |
| Sharing a working demo | Spaces | optional |

Today's notebook is the map — the pointers above are the territory.

## 9. Mini-lab — build a tiny evaluation pipeline end to end

Pick **one task** (other than sentiment analysis) and **one dataset** whose labels match that task, then
repeat the section-5 workflow on your own choice:

1. use `list_models(task=..., sort="downloads", direction=-1, limit=5)` to shortlist a model;
2. use `huggingface_hub.list_datasets(task_categories=..., search=..., limit=5)` to shortlist a dataset;
3. load a small slice of the dataset (`split="test[:50]"` or similar);
4. run the model over it with `pipeline()`;
5. score the predictions with `evaluate`;
6. write two sentences: was the score better or worse than you expected, and why?

### Starter code

In [ ]:
from huggingface_hub import list_datasets

# TODO: pick a task, e.g. "text-classification", "summarization", "question-answering"
TASK = "text-classification"

candidate_models = list(list_models(task=TASK, sort="downloads", direction=-1, limit=5))
candidate_datasets = list(list_datasets(task_categories=TASK, sort="downloads", direction=-1, limit=5))

print("Candidate models:")
for m in candidate_models:
    print(" -", m.id)

print("\nCandidate datasets:")
for d in candidate_datasets:
    print(" -", d.id)

# TODO: load your chosen model with pipeline(...), your chosen dataset with load_dataset(...),
# run predictions, and score them with evaluate.load(...).compute(...)


### Code walkthrough — mini-lab discovery arguments

```python
TASK = "text-classification"
```

`TASK` is deliberately a variable: students can change one value and reuse it in the model/dataset discovery calls.

```python
candidate_models = list(
    list_models(task=TASK, sort="downloads", direction=-1, limit=5)
)
```

Arguments are the same discovery controls used earlier:

- **`task=TASK`**: filter models by the selected task.
- **`sort="downloads"`**: rank by download count.
- **`direction=-1`**: descending order.
- **`limit=5`**: keep only a small shortlist.
- **outer `list(...)`**: materializes the iterable returned by `list_models()` so the results can be stored and reused.

```python
candidate_datasets = list(
    list_datasets(task_categories=TASK, sort="downloads", direction=-1, limit=5)
)
```

- **`task_categories=TASK`**: requests datasets associated with the selected task category in the Hub metadata used by this tutorial.
- **`sort`, `direction`, `limit`**: same ranking/shortlisting role as above.
- **outer `list(...)`**: consumes the returned iterable and keeps the results in memory.

The loops print **`.id`**, the Hub repository identifier for each candidate.

The final TODO is intentionally open-ended, but the expected pipeline is:

1. choose a model ID;
2. choose a compatible dataset and split;
3. construct `pipeline(...)`;
4. run predictions;
5. convert outputs into the label/answer format required by the dataset;
6. call `evaluate.load(metric).compute(...)`.

**Compatibility matters more than popularity:** the model's task, label space, language, and output format must match the dataset and metric.


## Glossary — quick reference

| Term | Meaning |
|---|---|
| **Hub** | Hugging Face's hosting service for models, datasets, and Spaces. |
| **Model card** | The `README.md` in a model repo — license, intended use, limitations, metrics. |
| **`pipeline()`** | High-level wrapper: tokenizer + model + post-processing in one call, for one task. |
| **`datasets.Dataset`** | A memory-mapped table of examples with typed `features` (e.g. `ClassLabel`). |
| **`evaluate`** | A library of standard metrics (accuracy, F1, BLEU, ROUGE, ...) with a uniform `.compute()` API. |
| **`accelerate`** | Runs the same training/inference code on CPU, single GPU, multi-GPU, or TPU without rewrites. |
| **Space** | A hosted demo app (often Gradio) that wraps a model behind a browser UI. |
| **Gated model** | Requires accepting a license on the model page before you can download weights. |
| **In- vs out-of-distribution evaluation** | Testing on data similar to training data vs. testing on data that differs — scores can drop a lot on the latter. |

---
